# Entrenamiento y Optimización de CodeT5 para comandos Bash

Este notebook automatiza el proceso de ajuste fino (fine-tuning) y optimización de un modelo de lenguaje para traducir lenguaje natural en español a comandos de terminal.

**Funciones principales:**
* **Entrenamiento:** Realiza el ajuste fino del modelo base `Salesforce/codet5-small` utilizando un dataset personalizado para tareas de traducción texto-a-código.
* **Exportación ONNX:** Convierte el modelo resultante al estándar ONNX para mejorar la portabilidad y compatibilidad de inferencia.
* **Cuantización Dinámica:** Optimiza los pesos del modelo ONNX para reducir su consumo de memoria y acelerar drásticamente la ejecución en CPU.
* **Prueba Interactiva:** Incluye un entorno de prueba en terminal para validar la inferencia del modelo optimizado inyectando variables de contexto.

**Requisitos de Hardware y Entorno:**
* **GPU (Para entrenamiento):** Se requiere una GPU NVIDIA con soporte para precisión mixta (FP16) y suficiente VRAM (ej. RTX serie 4000/5000 o equivalente).
* **CPU (Para inferencia y cuantización):** Un procesador moderno compatible con el conjunto de instrucciones AVX2.
* **Sistema:** Compatible nativamente con Linux. Requiere disponer de herramientas de compilación básicas y Rust en el sistema para las dependencias de los tokenizadores de HuggingFace.
* **Modelos que usan este Notebook**: https://huggingface.co/collections/jrodriiguezg/grape-models

**Dependencias de Python (pip):**
Se recomiendan los siguientes paquetes para ejecutar todas las celdas correctamente:
* `transformers==4.44.0` (definido explícitamente en el código)
* `datasets` (para la carga y partición del corpus)
* `torch` (backend necesario para el entrenamiento con Seq2SeqTrainer)
* `optimum[onnxruntime]` (para la exportación a ONNX y la cuantización dinámica)

In [ ]:
# <------------------------ Instalar Dependencias -------------------> 
!rustup default stable
!pip install "transformers==4.44.0" datasets torch "optimum[onnxruntime]"

In [ ]:

import os
from datasets import load_dataset
from transformers import AutoTokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Configuracion de rutas y modelo
MODEL_NAME = "Salesforce/codet5-small"
DATA_PATH = "./HF/chandonay-nl2bash-es/chandonay-nl2bash-es.json" # <------------------- Actualizar con la ruta del Dataset
OUT_DIR = "./grape-chandonay" # <---------------------------- Actualizar con directorio de salida del modelo

# Cargar tokenizador y modelo preentrenado
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

# Cargar dataset y crear particiones de entrenamiento y validacion (90/10)
dataset = load_dataset('json', data_files=DATA_PATH)
dataset = dataset["train"].train_test_split(test_size=0.1)

def preprocess_function(examples):
    # Definir la instruccion para guiar al modelo
    inputs = ["translate Spanish to Bash: " + doc for doc in examples["instruction"]]

    # Tokenizar los textos de entrada
    model_inputs = tokenizer(inputs, max_length=128, truncation=True, padding="max_length")

    # Tokenizar los comandos esperados (etiquetas)
    labels = tokenizer(
        text_target=examples["cmd"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Procesar y tokenizar el dataset por lotes
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Configurar parametros del entrenamiento
training_args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=25,
    weight_decay=0.01,
    warmup_steps=500,
    save_total_limit=2,
    predict_with_generate=True,
    logging_steps=50,
    fp16=True, # Utiliza media precision aprovechando la RTX 5070
    push_to_hub=False,
)

# Inicializar la clase Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

# Iniciar el proceso de entrenamiento
print("Iniciando entrenamiento...")
trainer.train()

# Guardar los resultados en disco
print("Guardando modelo final...")
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("Entrenamiento completado.")

# Exportar a ONNX
La siguiente linea coge el modelo anteriormente generado y lo exporta a ONNX

In [ ]:
import os
import glob
from optimum.onnxruntime import ORTModelForSeq2SeqLM, ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from transformers import AutoTokenizer

# Definir directorio base
base_dir = "./grape-malbec" # <------------------------ Directorio de entrada, salida de la celda anterior
checkpoints = glob.glob(os.path.join(base_dir, "checkpoint-*"))

# Seleccionar el ultimo checkpoint disponible
if checkpoints:
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"Checkpoint detectado: {latest_checkpoint}")
    model_id = os.path.abspath(latest_checkpoint)
else:
    print("No se encontraron checkpoints. Buscando en la raiz...")
    if os.path.exists(os.path.join(base_dir, "config.json")):
        model_id = os.path.abspath(base_dir)
        print(f"Usando modelo final en: {model_id}")
    else:
        raise FileNotFoundError(f"No encuentro 'config.json' en {base_dir}.")

save_directory = "./grape-malbec-t5-onnx" # <--------------------- Directorio de salida del modelo ONNX

print(f"Exportando modelo desde: {model_id}")
print(f"Destino: {save_directory}")

# Exportar modelo a ONNX de forma automatica
try:
    model = ORTModelForSeq2SeqLM.from_pretrained(model_id, export=True)
    model.save_pretrained(save_directory)
    print("Exportacion a ONNX completada.")
except Exception as e:
    print(f"Error en exportacion: {e}")
    exit()

# Guardar tokenizador
try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.save_pretrained(save_directory)
    print("Tokenizador guardado.")
except Exception as e:
    print(f"Aviso: No se pudo guardar el tokenizador ({e})")

# Iniciar proceso de cuantizacion
print("Iniciando cuantizacion...")
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)

onnx_models = [
    "encoder_model.onnx",
    "decoder_model.onnx",
    "decoder_with_past_model.onnx"
]

# Cuantizar componentes del modelo individualmente
for model_file in onnx_models:
    print(f"Optimizando {model_file}...")
    try:
        file_path = os.path.join(save_directory, model_file)
        if not os.path.exists(file_path):
            print("Saltado. El archivo no existe.")
            continue

        quantizer = ORTQuantizer.from_pretrained(save_directory, file_name=model_file)
        quantizer.quantize(save_dir=save_directory, quantization_config=qconfig)
        print("Hecho.")
    except Exception as e:
        print(f"Error optimizando {model_file}: {e}")

print(f"Proceso completado. Modelo optimizado en: {save_directory}")

# Cuantificar
La celda inferior coge el modelo ONNX anterior y lo cuantifica

In [ ]:
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from optimum.onnxruntime import ORTQuantizer
import os

# Ruta de entrada con el modelo ONNX original
onnx_dir = "./grape-malbec-t5-onnx" # <--------------- Salida de la celda anterior, modelo en ONNX

# Ruta de salida local
output_dir = "./outputs/grape-malbec-t5-quantized"  # <------------- Ruta de salida del modelo quantificado
os.makedirs(output_dir, exist_ok=True)

print(f"Los resultados se guardaran en: {output_dir}")
print(f"Buscando archivos originales en: {os.path.abspath(onnx_dir)}")

# Verificar que el modelo principal exista antes de iniciar
if not os.path.exists(os.path.join(onnx_dir, "encoder_model.onnx")):
    raise FileNotFoundError(f"No se encuentra 'encoder_model.onnx' en {onnx_dir}")

print("Iniciando configuracion de cuantizacion...")

# Configuracion de cuantizacion dinamica optimizada para CPU
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)

files_to_quantize = [
    "encoder_model.onnx",
    "decoder_model.onnx",
    "decoder_with_past_model.onnx"
]

# Cuantizar y guardar cada archivo individualmente
for file_name in files_to_quantize:
    print(f"Procesando {file_name}...")
    try:
        # Cargar modelo original desde el directorio local
        quantizer = ORTQuantizer.from_pretrained(onnx_dir, file_name=file_name)

        # Aplicar cuantizacion y guardar en el nuevo directorio
        quantizer.quantize(save_dir=output_dir, quantization_config=qconfig)
        print(f"Guardado correctamente: {file_name}")

    except Exception as e:
        print(f"Error procesando {file_name}: {e}")

print(f"Proceso completado. Modelos disponibles en: {output_dir}")

# Interferencia 
La celda inferior sirve para probar la interferencia del modelo generado

In [ ]:

import os
import sys
import time
from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer

# ==========================================
#  COLORES PARA LA TERMINAL
# ==========================================
class C:
    HEADER = '\033[95m'
    BLUE = '\033[94m'
    CYAN = '\033[96m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    END = '\033[0m'
    BOLD = '\033[1m'

# ==========================================
#  CONFIGURACIÓN
# ==========================================
MODEL_PATH = "./grape-malbec-t5-onnx" # <----------------- Modelo a usar, formato ONNX

#  CONTEXTO DE PRUEBA
# Déjalo en "[]" para pruebas normales.
# Cámbialo a "['web=10.0.0.5']" para probar SSH/Redes inteligentes.
# Descomentar los contextos segun el modelo a cargar
CURRENT_DIR = "/home/juan/Documents"
#
#TEST_CONTEXT = "['router=1.1.1.1', 'switch=2.2.2.2', 'ap=3.3.3.3']"
TEST_CONTEXT = "['pwd=/var/log', 'ls=syslog, NeoCore.py']"

# Variables globales
MODEL = None
TOKENIZER = None

# ==========================================
#  1. CARGADOR DEL MODELO (CORREGIDO)
# ==========================================
def load_single_model():
    global TOKENIZER, MODEL
    print(f"{C.HEADER} INICIANDO TEST UNITARIO (ONNX/CPU){C.END}")
    print(f" Objetivo: {C.BOLD}{MODEL_PATH}{C.END}")
    print(f" Contexto Simulado: {C.CYAN}{TEST_CONTEXT}{C.END}")
    print("-" * 50)

    # 1. Cargar Tokenizer
    try:
        print(f" Cargando Tokenizer...", end=" ", flush=True)
        try:
            TOKENIZER = AutoTokenizer.from_pretrained(MODEL_PATH)
        except:
            print(f"{C.YELLOW}(Fallback Flan-T5){C.END}", end=" ")
            TOKENIZER = AutoTokenizer.from_pretrained("Salesforce/codet5-small")
        print(f"{C.GREEN}OK{C.END}")
    except Exception as e:
        print(f"\n{C.RED} ERROR TOKENIZER: {e}{C.END}")
        sys.exit(1)

    # 2. Cargar el Modelo
    try:
        print(f" Cargando Modelo ONNX...", end=" ", flush=True)
        if not os.path.exists(MODEL_PATH):
            print(f"\n{C.RED} ERROR: No existe la ruta '{MODEL_PATH}'{C.END}")
            sys.exit(1)

        # Especificamos los nombres de archivo que aparecieron en tu error anterior
        # Usamos los _quantized porque son más rápidos.
        MODEL = ORTModelForSeq2SeqLM.from_pretrained(
            MODEL_PATH,
            encoder_file_name="encoder_model_quantized.onnx",
            decoder_file_name="decoder_model_quantized.onnx",
            decoder_with_past_file_name="decoder_model_quantized.onnx",
            use_cache=False
        )
        print(f"{C.GREEN}LISTO{C.END}")

    except Exception as e:
        print(f"\n{C.RED} ERROR MODELO: {str(e)}{C.END}")
        # Tip extra para debug
        print(f"{C.YELLOW}Tip: Verifica que los archivos .onnx estén dentro de {MODEL_PATH}{C.END}")
        sys.exit(1)

    print(f"\n{C.GREEN} Sistema Listo.{C.END}")

# ==========================================
#  2. BUCLE DE PRUEBA
# ==========================================
def main_loop():
    print("-" * 50)
    print(f"Escribe {C.BOLD}'salir'{C.END} para cerrar.")

    while True:
        try:
            user_input = input(f"\n{C.BLUE}Mango ❯{C.END} ").strip()

            if user_input.lower() in ['salir', 'exit', 'quit']:
                print(" Bye!")
                break

            if not user_input: continue

            # =====================================================
            #  AQUÍ ESTÁ LA CORRECCIÓN MÁGICA
            # =====================================================
            # Inyectamos la estructura que el modelo aprendió
            # Si el usuario escribe: "docker ubuntu"
            # El modelo recibe: "Contexto: [] | docker ubuntu"

            full_prompt = f"Contexto: {TEST_CONTEXT} | {user_input}"

            # (Opcional) Ver lo que realmente entra al modelo para debug
            # print(f"{C.YELLOW}DEBUG ENTRADA: {full_prompt}{C.END}")
            # =====================================================

            start_t = time.time()

            inputs = TOKENIZER(full_prompt, return_tensors="pt")

            # max_length=128 es suficiente para comandos
            outputs = MODEL.generate(**inputs, max_length=128)
            response = TOKENIZER.decode(outputs[0], skip_special_tokens=True)

            end_t = time.time()
            latency = (end_t - start_t) * 1000

            print(f"   ⏱  {latency:.0f}ms")
            print(f"   {C.GREEN}➜ {response}{C.END}")

        except KeyboardInterrupt:
            print("\n Cancelado.")
            break
        except Exception as e:
            print(f"{C.RED} Error: {e}{C.END}")

if __name__ == "__main__":
    load_single_model()
    main_loop()